In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/processed/energy_features_2019.csv')

In [ ]:
df.head(6)

In [ ]:
RANDOM_STATE = 42
df.sample(10, random_state = RANDOM_STATE)

In [ ]:
df.avg_energy.values

In [ ]:
average_energy_demand = df.avg_energy.values
average_temperature = df.temperature.values

In [ ]:
def annual_daily_average(arr:np.typing.NDArray, timestamp = 24) -> np.typing.NDArray:
    
    annual_arr = []
    length_arr = len(arr)
    ele_sum = 0
    
    for ele in range(length_arr):
        
        if (ele+1) % (timestamp) == 0:
            ele_sum += arr[ele]
            annual_arr.append(ele_sum)
            ele_sum = 0
        else:
            ele_sum += arr[ele]

    return np.array(annual_arr)
    
    

In [ ]:
annual_daily_energy_demand = annual_daily_average(average_energy_demand)

In [ ]:
len(annual_daily_energy_demand)

In [ ]:
annual_daily_energy_demand[0]

In [ ]:
df.head(24).sum()

In [ ]:
annual_daily_temp =  annual_daily_average(average_temperature)

In [ ]:
annual_daily_temp[0]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
normalized_annual_temp = scaler.fit_transform(annual_daily_temp.reshape(-1,1))
normalized_annual_demand = scaler.fit_transform(annual_daily_energy_demand.reshape(-1,1))
print(len(normalized_annual_demand))
print(len(normalized_annual_temp))

In [ ]:
#  Air temperature and energy demand annual trends
plt.figure(figsize=(8,5), dpi = 150)
plt.plot(normalized_annual_demand, linewidth = 2, label= 'Energy Demand', alpha = 0.9)
plt.plot(normalized_annual_temp, linewidth = 2,label = "Air temperature", linestyle = '--', alpha = 0.9)
plt.ylabel('Normalized Dimension Less Value')
plt.xlabel('Day')
plt.xticks(np.arange(0, 400, 37))
plt.yticks(np.arange(0,1.4, 0.2))
plt.grid()
plt.legend(loc = 1)
plt.tight_layout()
plt.savefig('../results/temp_energy_relation.png', dpi = 200)

In [ ]:
annual_daily_energy_demand
threshold = 22.1918
low = 0.0941
high = 0.141
energy_price = []
energy_cost = []
for day in range(len(annual_daily_energy_demand)):
    energy = annual_daily_energy_demand[day] 
    if energy > threshold:
        energy_price.append(high)
        energy_cost.append(high*(energy-threshold) + low*threshold)

    else:
        energy_price.append(low)
        energy_cost.append(low*energy)

energy_cost = np.array(energy_cost)
energy_price = np.array(energy_price)




In [ ]:
#  energy relation
fig, (ax1, ax2, ax3) = plt.subplots(3, 1)

ax1.plot(annual_daily_energy_demand, linewidth = 2, label = 'annual demand')
ax1.legend()
ax1.grid()
ax1.set_xticks(np.arange(0, 400, 37))
ax1.set_yticks(np.arange(0, max(annual_daily_energy_demand),  int(max(annual_daily_energy_demand)/5 )) )


ax2.plot(energy_price, linewidth = 2, label = 'energy_price')
ax2.legend()
ax2.grid()
ax2.set_xticks(np.arange(0, 400, 37))


ax3.plot(energy_cost, linewidth = 2, label = 'energy_cost')
ax3.legend()
ax3.grid()
ax3.set_xticks(np.arange(0, 400, 37))
ax3.set_yticks(np.arange(0, 8, 2))
plt.tight_layout() 
fig.savefig('../results/energy_pricing.png',dpi = 200)